Kaggle - 2008-2025 NBA dataset. 
This project will investigate betting odds(spreads, ML) of the listed games.

In [15]:
import pandas as pd
df = pd.read_csv("../archive.zip")

In [16]:
df.head(10)

,season,date,regular,playoffs,away,home,score_away,score_home,q1_away,q2_away,...,ot_home,whos_favored,spread,total,moneyline_away,moneyline_home,h2_spread,h2_total,id_spread,id_total
0,2008,2007-10-30,True,False,por,sa,97,106,26,23,...,0,home,13.0,189.5,900.0,-1400.0,5.0,95.0,0.0,1
1,2008,2007-10-30,True,False,utah,gs,117,96,28,34,...,0,home,1.0,212.0,100.0,-120.0,3.0,105.5,0.0,1
2,2008,2007-10-30,True,False,hou,lal,95,93,16,27,...,0,away,5.0,199.0,-230.0,190.0,3.0,99.0,0.0,0
3,2008,2007-10-31,True,False,phi,tor,97,106,22,28,...,0,home,6.5,191.0,255.0,-305.0,2.0,96.5,1.0,1
4,2008,2007-10-31,True,False,wsh,ind,110,119,23,22,...,16,away,1.5,203.5,-125.0,105.0,1.0,105.0,0.0,1
5,2008,2007-10-31,True,False,mil,orl,83,102,24,22,...,0,home,6.5,197.0,245.0,-295.0,3.5,97.0,1.0,0
6,2008,2007-10-31,True,False,chi,bkn,103,112,21,20,...,16,home,1.5,186.0,105.0,-125.0,3.0,94.0,1.0,1
7,2008,2007-10-31,True,False,dal,cle,92,74,29,25,...,0,away,2.5,184.0,-140.0,120.0,4.0,91.5,1.0,0
8,2008,2007-10-31,True,False,sa,mem,104,101,23,29,...,0,away,6.5,201.0,-250.0,210.0,4.0,99.5,0.0,1
9,2008,2007-10-31,True,False,sac,no,90,104,24,18,...,0,home,9.5,190.5,425.0,-525.0,1.5,94.0,1.0,1


Investigate the data to determine what is shown to see if more columns need to be added.

In [17]:
df.dtypes

season              int64
date               object
regular              bool
playoffs             bool
away               object
home               object
score_away          int64
score_home          int64
q1_away             int64
q2_away             int64
q3_away             int64
q4_away             int64
ot_away             int64
q1_home             int64
q2_home             int64
q3_home             int64
q4_home             int64
ot_home             int64
whos_favored       object
spread            float64
total             float64
moneyline_away    float64
moneyline_home    float64
h2_spread         float64
h2_total          float64
id_spread         float64
id_total            int64
dtype: object

In [18]:
df.shape

(23118, 27)

Now lets divide the outcomes of the games in relation to the spread. 
There are 4 possible outcomes we are interested in:
    1. UD Loses - Covers Spread
    2. UD Wins
    3. Fave Wins - Covers Spread
    4. PUSH

In [19]:
#Function to determine if the Underdog wins
def UD_win(row):
    if row["whos_favored"] == "away":
        return row["score_home"] > row["score_away"]

    elif row["whos_favored"] == "home":
        return row["score_away"] > row["score_home"]

In [20]:
df["underdog_wins"] = df.apply(UD_win, axis = 1)
df.head(10)

,season,date,regular,playoffs,away,home,score_away,score_home,q1_away,q2_away,...,whos_favored,spread,total,moneyline_away,moneyline_home,h2_spread,h2_total,id_spread,id_total,underdog_wins
0,2008,2007-10-30,True,False,por,sa,97,106,26,23,...,home,13.0,189.5,900.0,-1400.0,5.0,95.0,0.0,1,False
1,2008,2007-10-30,True,False,utah,gs,117,96,28,34,...,home,1.0,212.0,100.0,-120.0,3.0,105.5,0.0,1,True
2,2008,2007-10-30,True,False,hou,lal,95,93,16,27,...,away,5.0,199.0,-230.0,190.0,3.0,99.0,0.0,0,False
3,2008,2007-10-31,True,False,phi,tor,97,106,22,28,...,home,6.5,191.0,255.0,-305.0,2.0,96.5,1.0,1,False
4,2008,2007-10-31,True,False,wsh,ind,110,119,23,22,...,away,1.5,203.5,-125.0,105.0,1.0,105.0,0.0,1,True
5,2008,2007-10-31,True,False,mil,orl,83,102,24,22,...,home,6.5,197.0,245.0,-295.0,3.5,97.0,1.0,0,False
6,2008,2007-10-31,True,False,chi,bkn,103,112,21,20,...,home,1.5,186.0,105.0,-125.0,3.0,94.0,1.0,1,False
7,2008,2007-10-31,True,False,dal,cle,92,74,29,25,...,away,2.5,184.0,-140.0,120.0,4.0,91.5,1.0,0,False
8,2008,2007-10-31,True,False,sa,mem,104,101,23,29,...,away,6.5,201.0,-250.0,210.0,4.0,99.5,0.0,1,False
9,2008,2007-10-31,True,False,sac,no,90,104,24,18,...,home,9.5,190.5,425.0,-525.0,1.5,94.0,1.0,1,False


In [21]:
#Function to determine if UD Covers, but doesnt win
def UD_covers_no_win(row):
    if (row["whos_favored"] == "away") :
        if (row["score_away"] - row["spread"] == row["score_home"]):
            return "Push"
        if (row["underdog_wins"] == True):
            return False
        return row["score_away"] - row["spread"] < row["score_home"]
    elif (row["whos_favored"] == "home") :
        if (row["score_home"] - row["spread"] == row["score_away"]):
            return "Push"
        if (row["underdog_wins"] == True):
            return False
        return row["score_home"] - row["spread"] < row["score_away"]
    

In [22]:
df["underdog_covers_no_win"] = df.apply(UD_covers_no_win, axis = 1)
df.head(10)

,season,date,regular,playoffs,away,home,score_away,score_home,q1_away,q2_away,...,spread,total,moneyline_away,moneyline_home,h2_spread,h2_total,id_spread,id_total,underdog_wins,underdog_covers_no_win
0,2008,2007-10-30,True,False,por,sa,97,106,26,23,...,13.0,189.5,900.0,-1400.0,5.0,95.0,0.0,1,False,True
1,2008,2007-10-30,True,False,utah,gs,117,96,28,34,...,1.0,212.0,100.0,-120.0,3.0,105.5,0.0,1,True,False
2,2008,2007-10-30,True,False,hou,lal,95,93,16,27,...,5.0,199.0,-230.0,190.0,3.0,99.0,0.0,0,False,True
3,2008,2007-10-31,True,False,phi,tor,97,106,22,28,...,6.5,191.0,255.0,-305.0,2.0,96.5,1.0,1,False,False
4,2008,2007-10-31,True,False,wsh,ind,110,119,23,22,...,1.5,203.5,-125.0,105.0,1.0,105.0,0.0,1,True,False
5,2008,2007-10-31,True,False,mil,orl,83,102,24,22,...,6.5,197.0,245.0,-295.0,3.5,97.0,1.0,0,False,False
6,2008,2007-10-31,True,False,chi,bkn,103,112,21,20,...,1.5,186.0,105.0,-125.0,3.0,94.0,1.0,1,False,False
7,2008,2007-10-31,True,False,dal,cle,92,74,29,25,...,2.5,184.0,-140.0,120.0,4.0,91.5,1.0,0,False,False
8,2008,2007-10-31,True,False,sa,mem,104,101,23,29,...,6.5,201.0,-250.0,210.0,4.0,99.5,0.0,1,False,True
9,2008,2007-10-31,True,False,sac,no,90,104,24,18,...,9.5,190.5,425.0,-525.0,1.5,94.0,1.0,1,False,False


In [23]:
#Favorite Wins and covers
def fave_cover(row):
    if row["whos_favored"] == "away":
        diff = row["score_away"] - row["spread"]
        if diff == 0:
            return "PUSH"
        else:
            return row["score_away"] - row["spread"] > row["score_home"]

    elif row["whos_favored"] == "home":
        diff = row["score_home"] - row["spread"]
        if diff == 0:
            return "PUSH"
        else:
            return row["score_home"] - row["spread"] > row["score_away"]

In [24]:
df["fave_covers"] = df.apply(fave_cover, axis = 1)
df.head(10)

,season,date,regular,playoffs,away,home,score_away,score_home,q1_away,q2_away,...,total,moneyline_away,moneyline_home,h2_spread,h2_total,id_spread,id_total,underdog_wins,underdog_covers_no_win,fave_covers
0,2008,2007-10-30,True,False,por,sa,97,106,26,23,...,189.5,900.0,-1400.0,5.0,95.0,0.0,1,False,True,False
1,2008,2007-10-30,True,False,utah,gs,117,96,28,34,...,212.0,100.0,-120.0,3.0,105.5,0.0,1,True,False,False
2,2008,2007-10-30,True,False,hou,lal,95,93,16,27,...,199.0,-230.0,190.0,3.0,99.0,0.0,0,False,True,False
3,2008,2007-10-31,True,False,phi,tor,97,106,22,28,...,191.0,255.0,-305.0,2.0,96.5,1.0,1,False,False,True
4,2008,2007-10-31,True,False,wsh,ind,110,119,23,22,...,203.5,-125.0,105.0,1.0,105.0,0.0,1,True,False,False
5,2008,2007-10-31,True,False,mil,orl,83,102,24,22,...,197.0,245.0,-295.0,3.5,97.0,1.0,0,False,False,True
6,2008,2007-10-31,True,False,chi,bkn,103,112,21,20,...,186.0,105.0,-125.0,3.0,94.0,1.0,1,False,False,True
7,2008,2007-10-31,True,False,dal,cle,92,74,29,25,...,184.0,-140.0,120.0,4.0,91.5,1.0,0,False,False,True
8,2008,2007-10-31,True,False,sa,mem,104,101,23,29,...,201.0,-250.0,210.0,4.0,99.5,0.0,1,False,True,False
9,2008,2007-10-31,True,False,sac,no,90,104,24,18,...,190.5,425.0,-525.0,1.5,94.0,1.0,1,False,False,True


In [25]:
def determine_outcome(row):
    if row["underdog_wins"]:
        return "Underdog Wins"
    elif row["fave_covers"]:
        return "Favorite Covers"
    elif row["underdog_covers_no_win"] == "Push":
        return "Push"
    else:
        return "Underdog Covers But Loses"
    
    
df["ATS_outcome"] = df.apply(determine_outcome, axis=1)

In [26]:
def season_label(row):
    if row["regular"]:
        return "Regular Season"
    else:
        if row["playoffs"]:
            return "Playoffs"
        else:
            return "PlayIn"

df["Season_Type"] = df.apply(season_label, axis = 1)

In [27]:
df.dropna(subset=["spread"], inplace = True)
df.to_csv('df_with_categorization.csv',index = False)